# Voting Regression

In [127]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

## Ler Dados

In [128]:
df_costs = pd.read_csv("./dataset/employees.csv")
df_costs.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              1338 non-null   int64  
 1   sex              1338 non-null   str    
 2   bmi              1338 non-null   float64
 3   children         1338 non-null   int64  
 4   smoker           1338 non-null   int64  
 5   region           1338 non-null   str    
 6   medical charges  1338 non-null   float64
dtypes: float64(2), int64(3), str(2)
memory usage: 73.3 KB


In [129]:
df_costs.tail(10)

,age,sex,bmi,children,smoker,region,medical charges
1328,23,female,24.225,2,0,northeast,22395.74424
1329,52,male,38.600,2,0,southwest,10325.20600
1330,57,female,25.740,2,0,southeast,12629.16560
1331,23,female,33.400,0,0,southwest,10795.93733
1332,52,female,44.700,3,0,southwest,11411.68500
1333,50,male,30.970,3,0,northwest,10600.54830
1334,18,female,31.920,0,0,northeast,2205.98080
1335,18,female,36.850,0,0,southeast,1629.83350
1336,21,female,25.800,0,0,southwest,2007.94500
1337,61,female,29.070,0,1,northwest,29141.36030


## Preparação dos Dados

In [130]:
X = df_costs.drop(columns=['medical charges'])
y = df_costs['medical charges']

In [131]:
preprocessor: ColumnTransformer = joblib.load("preprocessor.pkl")

In [132]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=51)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.fit_transform(X_test)

In [133]:
X_test.shape

(268, 10)

In [134]:
X_train.shape

(1070, 10)

## Treinamento do Modelo

In [135]:
linear_model = LinearRegression()
elastic_model = ElasticNet(random_state=51)
tree_model = DecisionTreeRegressor(random_state=51)

voting_model = VotingRegressor(
  estimators=[
    ('linear', linear_model),
    ('elastic', elastic_model),
    ('tree', tree_model),
  ],
  weights=[1, 1, 1],
  verbose=True,
)

In [136]:
voting_model.fit(X=X_train, y=y_train)

[Voting] ................... (1 of 3) Processing linear, total=   0.0s
[Voting] .................. (2 of 3) Processing elastic, total=   0.0s
[Voting] ..................... (3 of 3) Processing tree, total=   0.0s


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('linear', ...), ('elastic', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.","[1, 1, ...]"
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
estimators_ estimators_: list of regressorsThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'.,list,"[LinearRegression(), ElasticNet(random_state=51), DecisionTreeR...ndom_state=51)]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,10
named_estimators_ named_estimators_: :class:`~sklearn.utils.Bunch`Attribute to access any fitted sub-estimators by name... versionadded:: 0.20,Bunch,{'linear': Li...dom_state=51)}
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06


## Análise de Métricas

In [137]:
y_pred = voting_model.predict(X=X_test)

In [138]:
mae = mean_absolute_error(y_pred=y_pred, y_true=y_test)
rmse = root_mean_squared_error(y_pred=y_pred, y_true=y_test)
r2 = r2_score(y_pred=y_pred, y_true=y_test)

In [139]:
print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R2-Score: {r2}")

Mean Absolute Error: 4115.146387511171
Root Mean Squared Error: 6313.4659406646715
R2-Score: 0.7707656796451199


## Calcular importâncias das variáveis

In [140]:
importances = list()

for estimador in voting_model.estimators_:
    if hasattr(estimador, 'coef_'):
        importances.append(np.abs(estimador.coef_))
    elif hasattr(estimador, 'feature_importances_'):
        importances.append(estimador.feature_importances_)
    else:
        print(f"Não foi possível carregar importância do modelo {type(estimador).__name__}")

In [141]:
importances[2].shape

(10,)

In [142]:
importancia_media = np.mean(importances, axis=0)

In [143]:
importances_percentage = importancia_media / np.sum(importancia_media)

In [144]:
feature_names = preprocessor.get_feature_names_out()

In [145]:
df_features = pd.DataFrame({ 'feature': feature_names, 'importance': importances_percentage})
df_features.sort_values(by='importance', ascending=True, inplace=True)

In [146]:
px.bar(
  df_features,
  x='importance',
  y='feature'
)

# Mostrar como funciona o modelo

In [147]:
X_sample = X_test[2].reshape(1, -1)

linear_pred = voting_model.named_estimators_['linear'].predict(X_sample)
elastic_pred = voting_model.named_estimators_['elastic'].predict(X_sample)
tree_pred = voting_model.named_estimators_['tree'].predict(X_sample)

final_pred = voting_model.predict(X_sample)

result = (linear_pred[0] + elastic_pred[0] + tree_pred[0])

In [148]:
print(f"Linear Regression: {linear_pred}")
print(f"Elastic Net: {elastic_pred}")
print(f"Decision Tree Regressor: {tree_pred}")
print(f"Média dos 3 Acima: {result}")
print(f"Voting Regressor: {final_pred}")

Linear Regression: [2154.54735778]
Elastic Net: [6101.48508719]
Decision Tree Regressor: [25081.76784]
Média dos 3 Acima: 33337.800284975754
Voting Regressor: [11112.60009499]
